In [2]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

import os

files = [
    "/content/drive/MyDrive/sql_create_context_v4.json",
    "/content/drive/MyDrive/train-00000-of-00001.parquet",
    "/content/drive/MyDrive/validation-00000-of-00001.parquet",
    "/content/drive/MyDrive/test-00000-of-00001.parquet"
]

for file in files:
    print(os.path.basename(file), "->", os.path.exists(file))


sql_create_context_v4.json -> True
train-00000-of-00001.parquet -> True
validation-00000-of-00001.parquet -> True
test-00000-of-00001.parquet -> True


In [4]:

import pandas as pd

sql_df = pd.read_json(
    "/content/drive/MyDrive/sql_create_context_v4.json"
)

clinc_train_df = pd.read_parquet(
    "/content/drive/MyDrive/train-00000-of-00001.parquet"
)

clinc_validation_df = pd.read_parquet(
    "/content/drive/MyDrive/validation-00000-of-00001.parquet"
)

clinc_test_df = pd.read_parquet(
    "/content/drive/MyDrive/test-00000-of-00001.parquet"
)

print("SQL dataset:", sql_df.shape)
print("CLINC train:", clinc_train_df.shape)
print("CLINC validation:", clinc_validation_df.shape)
print("CLINC test:", clinc_test_df.shape)

print("\nSQL columns:")
print(sql_df.columns.tolist())

print("\nCLINC columns:")
print(clinc_train_df.columns.tolist())


SQL dataset: (78577, 3)
CLINC train: (15250, 2)
CLINC validation: (3100, 2)
CLINC test: (5500, 2)

SQL columns:
['question', 'context', 'answer']

CLINC columns:
['text', 'intent']


In [5]:

from datasets import load_dataset

clinc = load_dataset("clinc/clinc_oos", "plus")

intent_feature = clinc["train"].features["intent"]

for df in [clinc_train_df, clinc_validation_df, clinc_test_df]:
    df["intent_name"] = df["intent"].apply(intent_feature.int2str)

general_intents = [
    "translate",
    "definition",
    "meaning_of_life",
    "fun_fact",
    "time",
    "distance",
    "next_song",
    "calculator",
    "thank_you",
    "calories",
    "goodbye",
    "what_song",
    "measurement_conversion",
    "weather",
    "directions",
    "greeting",
    "play_music",
    "tell_joke",
    "recipe",
    "traffic"
]

print("Selected general intents:", len(general_intents))
print(clinc_train_df["intent_name"].head())


README.md:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

plus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  312kB            

plus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plus/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 77.8kB            

plus/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plus/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

Selected general intents: 20
0    translate
1    translate
2    translate
3    translate
4    translate
Name: intent_name, dtype: object


In [6]:

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Clean SQL dataset
sql_clean = sql_df[["question"]].copy()
sql_clean = sql_clean.dropna(subset=["question"])
sql_clean["text"] = sql_clean["question"].astype(str).str.strip()
sql_clean = sql_clean[sql_clean["text"] != ""]
sql_clean = sql_clean.drop_duplicates(subset=["text"])

# 2. Split unique SQL questions to prevent SQL-question overlap
sql_train_pool, sql_temp_pool = train_test_split(
    sql_clean, test_size=0.20, random_state=42
)

sql_val_pool, sql_test_pool = train_test_split(
    sql_temp_pool, test_size=0.50, random_state=42
)

# 3. Select SQL examples
sql_train = sql_train_pool.sample(n=2000, random_state=42)
sql_val = sql_val_pool.sample(n=400, random_state=42)
sql_test = sql_test_pool.sample(n=600, random_state=42)

sql_train = sql_train[["text"]].copy()
sql_val = sql_val[["text"]].copy()
sql_test = sql_test[["text"]].copy()

sql_train["label"] = "DATABASE"
sql_val["label"] = "DATABASE"
sql_test["label"] = "DATABASE"

# 4. Select general queries from the respective CLINC splits
def prepare_general(df, sample_size, seed):
    general = df[df["intent_name"].isin(general_intents)].copy()
    general = general.dropna(subset=["text"])
    general["text"] = general["text"].astype(str).str.strip()
    general = general[general["text"] != ""]
    general = general.drop_duplicates(subset=["text"])

    if len(general) < sample_size:
        raise ValueError(
            f"Only {len(general)} general queries available; "
            f"{sample_size} required."
        )

    general = general.sample(n=sample_size, random_state=seed)
    general = general[["text"]].copy()
    general["label"] = "GENERAL"
    return general

general_train = prepare_general(clinc_train_df, 2000, 42)
general_val = prepare_general(clinc_validation_df, 400, 42)
general_test = prepare_general(clinc_test_df, 600, 42)

# 5. Combine and shuffle
train_df = pd.concat(
    [sql_train, general_train], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

validation_df = pd.concat(
    [sql_val, general_val], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

test_df = pd.concat(
    [sql_test, general_test], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

# 6. Save permanently to Google Drive
output_dir = "/content/drive/MyDrive/Task1_Research"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(f"{output_dir}/task1_train.csv", index=False)
validation_df.to_csv(
    f"{output_dir}/task1_validation.csv", index=False
)
test_df.to_csv(f"{output_dir}/task1_test.csv", index=False)

# 7. Verify
for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", validation_df),
    ("TEST", test_df)
]:
    print(f"\n{name} shape: {df.shape}")
    print(df["label"].value_counts())
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate texts:", df["text"].duplicated().sum())

print("\nSaved dataset files in:", output_dir)



TRAIN shape: (4000, 2)
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64
Missing values: 0
Duplicate texts: 0

VALIDATION shape: (800, 2)
label
GENERAL     400
DATABASE    400
Name: count, dtype: int64
Missing values: 0
Duplicate texts: 0

TEST shape: (1200, 2)
label
GENERAL     600
DATABASE    600
Name: count, dtype: int64
Missing values: 0
Duplicate texts: 0

Saved dataset files in: /content/drive/MyDrive/Task1_Research


In [7]:

import pandas as pd

output_dir = "/content/drive/MyDrive/Task1_Research"

train_df = pd.read_csv(f"{output_dir}/task1_train.csv")
validation_df = pd.read_csv(f"{output_dir}/task1_validation.csv")
test_df = pd.read_csv(f"{output_dir}/task1_test.csv")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

display(train_df.head())


Train: (4000, 2)
Validation: (800, 2)
Test: (1200, 2)


,text,label
0,how many coupless totalled 14.5,DATABASE
1,what are the atmospheric conditions in sparks,GENERAL
2,Can you tell me the highest Total votes that h...,DATABASE
3,what can you tell me about miami weather,GENERAL
4,i am needing to know the current time in the e...,GENERAL


In [8]:
sql_clean = sql_clean.drop_duplicates(subset=["text"])

In [9]:

train_texts = set(train_df["text"].str.strip().str.lower())
val_texts = set(validation_df["text"].str.strip().str.lower())
test_texts = set(test_df["text"].str.strip().str.lower())

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))


Train ∩ Validation: 1
Train ∩ Test: 0
Validation ∩ Test: 0


In [10]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            max_features=20000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

baseline_model.fit(train_df["text"], train_df["label"])

print("Baseline model training completed!")


Baseline model training completed!


In [11]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Validation evaluation
val_predictions = baseline_model.predict(validation_df["text"])

print("VALIDATION ACCURACY:")
print(accuracy_score(validation_df["label"], val_predictions))

print("\nVALIDATION CLASSIFICATION REPORT:")
print(classification_report(validation_df["label"], val_predictions))

print("\nVALIDATION CONFUSION MATRIX:")
print(confusion_matrix(
    validation_df["label"],
    val_predictions,
    labels=["DATABASE", "GENERAL"]
))

# Final test evaluation
test_predictions = baseline_model.predict(test_df["text"])

print("\nTEST ACCURACY:")
print(accuracy_score(test_df["label"], test_predictions))

print("\nTEST CLASSIFICATION REPORT:")
print(classification_report(test_df["label"], test_predictions))

print("\nTEST CONFUSION MATRIX:")
print(confusion_matrix(
    test_df["label"],
    test_predictions,
    labels=["DATABASE", "GENERAL"]
))


VALIDATION ACCURACY:
0.9775

VALIDATION CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    DATABASE       0.97      0.99      0.98       400
     GENERAL       0.99      0.97      0.98       400

    accuracy                           0.98       800
   macro avg       0.98      0.98      0.98       800
weighted avg       0.98      0.98      0.98       800


VALIDATION CONFUSION MATRIX:
[[395   5]
 [ 13 387]]

TEST ACCURACY:
0.9833333333333333

TEST CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    DATABASE       0.98      0.99      0.98       600
     GENERAL       0.99      0.97      0.98       600

    accuracy                           0.98      1200
   macro avg       0.98      0.98      0.98      1200
weighted avg       0.98      0.98      0.98      1200


TEST CONFUSION MATRIX:
[[595   5]
 [ 15 585]]


In [12]:

overlap = (
    set(train_df["text"].str.strip().str.lower())
    & set(validation_df["text"].str.strip().str.lower())
)

print("Number of overlapping queries:", len(overlap))
print("\nOverlapping query:")
for query in overlap:
    print(query)


Number of overlapping queries: 1

Overlapping query:
hey what's up


In [13]:

train_keys = set(train_df["text"].str.strip().str.lower())

validation_df = validation_df[
    ~validation_df["text"].str.strip().str.lower().isin(train_keys)
].reset_index(drop=True)

output_dir = "/content/drive/MyDrive/Task1_Research"

train_df.to_csv(f"{output_dir}/task1_train.csv", index=False)
validation_df.to_csv(
    f"{output_dir}/task1_validation.csv", index=False
)
test_df.to_csv(f"{output_dir}/task1_test.csv", index=False)

print("Updated validation shape:", validation_df.shape)
print("Train-validation overlap:",
      len(set(train_df["text"].str.strip().str.lower())
          & set(validation_df["text"].str.strip().str.lower())))
print("Train-test overlap:",
      len(set(train_df["text"].str.strip().str.lower())
          & set(test_df["text"].str.strip().str.lower())))
print("Validation-test overlap:",
      len(set(validation_df["text"].str.strip().str.lower())
          & set(test_df["text"].str.strip().str.lower())))


Updated validation shape: (799, 2)
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [14]:

print("Train labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(validation_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())



Train labels:
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64

Validation labels:
label
DATABASE    400
GENERAL     399
Name: count, dtype: int64

Test labels:
label
GENERAL     600
DATABASE    600
Name: count, dtype: int64


In [15]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(train_df["text"], train_df["label"])

print("Baseline model retrained successfully!")

# Validation evaluation
val_predictions = baseline_model.predict(validation_df["text"])

print("\nVALIDATION ACCURACY:",
      accuracy_score(validation_df["label"], val_predictions))

print("\nVALIDATION REPORT:")
print(classification_report(validation_df["label"], val_predictions))

# Test evaluation
test_predictions = baseline_model.predict(test_df["text"])

print("\nTEST ACCURACY:",
      accuracy_score(test_df["label"], test_predictions))

print("\nTEST REPORT:")
print(classification_report(test_df["label"], test_predictions))

print("\nTEST CONFUSION MATRIX:")
print(confusion_matrix(
    test_df["label"],
    test_predictions,
    labels=["DATABASE", "GENERAL"]
))


Baseline model retrained successfully!

VALIDATION ACCURACY: 0.9774718397997497

VALIDATION REPORT:
              precision    recall  f1-score   support

    DATABASE       0.97      0.99      0.98       400
     GENERAL       0.99      0.97      0.98       399

    accuracy                           0.98       799
   macro avg       0.98      0.98      0.98       799
weighted avg       0.98      0.98      0.98       799


TEST ACCURACY: 0.9833333333333333

TEST REPORT:
              precision    recall  f1-score   support

    DATABASE       0.98      0.99      0.98       600
     GENERAL       0.99      0.97      0.98       600

    accuracy                           0.98      1200
   macro avg       0.98      0.98      0.98      1200
weighted avg       0.98      0.98      0.98      1200


TEST CONFUSION MATRIX:
[[595   5]
 [ 15 585]]


In [16]:

import os
import json
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

output_dir = "/content/drive/MyDrive/Task1_Research"
os.makedirs(output_dir, exist_ok=True)

results = []

for split_name, actual, predicted in [
    ("Validation", validation_df["label"], val_predictions),
    ("Test", test_df["label"], test_predictions)
]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="macro", zero_division=0
    )

    results.append({
        "dataset": split_name,
        "accuracy": accuracy_score(actual, predicted),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "sample_count": len(actual)
    })

results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{output_dir}/task1_baseline_metrics.csv", index=False
)

joblib.dump(
    baseline_model,
    f"{output_dir}/task1_tfidf_logistic_baseline.joblib"
)

print(results_df.to_string(index=False))
print("\nSaved metrics and model to:", output_dir)


   dataset  accuracy  macro_precision  macro_recall  macro_f1  sample_count
Validation  0.977472         0.977675      0.977459  0.977469           799
      Test  0.983333         0.983468      0.983333  0.983332          1200

Saved metrics and model to: /content/drive/MyDrive/Task1_Research


In [17]:

import pandas as pd

vectorizer = baseline_model.named_steps["tfidf"]
classifier = baseline_model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]
classes = classifier.classes_

positive_idx = coefficients.argsort()[-20:][::-1]
negative_idx = coefficients.argsort()[:20]

print("Classifier classes:", classes)

print("\nTop features for", classes[1])
print(pd.DataFrame({
    "word_or_phrase": feature_names[positive_idx],
    "coefficient": coefficients[positive_idx]
}).to_string(index=False))

print("\nTop features for", classes[0])
print(pd.DataFrame({
    "word_or_phrase": feature_names[negative_idx],
    "coefficient": coefficients[negative_idx]
}).to_string(index=False))


Classifier classes: ['DATABASE' 'GENERAL']

Top features for GENERAL
word_or_phrase  coefficient
            to     5.098014
           you     4.712495
            me     3.584063
          song     3.196614
            it     2.988106
      calories     2.876200
          this     2.840672
       meaning     2.572623
        thanks     2.542110
         about     2.475134
       traffic     2.341045
      what the     2.330686
          life     2.204858
        are in     2.197105
            do     2.133815
       weather     2.120644
           how     2.090389
        please     2.067560
           now     2.008723
    meaning of     1.963248

Top features for DATABASE
word_or_phrase  coefficient
           has    -4.593606
          when    -4.467546
         which    -4.371781
           the    -4.031554
           was    -3.995418
            of    -3.217563
          than    -3.196455
          with    -3.100234
           and    -3.020865
           who    -2.793552
        

In [18]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [19]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [20]:

%pip install -q transformers accelerate sentencepiece scikit-learn


In [21]:

import torch
import pandas as pd
import os

output_dir = "/content/drive/MyDrive/Task1_Research"

train_df = pd.read_csv(f"{output_dir}/task1_train.csv")
validation_df = pd.read_csv(f"{output_dir}/task1_validation.csv")
test_df = pd.read_csv(f"{output_dir}/task1_test.csv")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nTrain:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())


GPU available: True
GPU: Tesla T4

Train: (4000, 2)
Validation: (799, 2)
Test: (1200, 2)

Train labels:
label
DATABASE    2000
GENERAL     2000
Name: count, dtype: int64


In [22]:

%pip install -q -U transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.6 MB/s eta 0:00:00


In [23]:

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ))


GPU available: True
GPU name: Tesla T4
GPU memory (GB): 14.56


In [24]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Load the model in 4-bit to reduce GPU memory usage
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading 7B teacher model...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

teacher_model.eval()

print("Teacher model loaded successfully!")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))


Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher model loaded successfully!
Model: Qwen/Qwen2.5-7B-Instruct
GPU: Tesla T4


In [25]:

def ask_teacher(query):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a query classification assistant. "
                "Classify each user query into exactly one category: "
                "DATABASE or GENERAL. "
                "DATABASE means the query concerns databases, SQL, "
                "tables, records, or database operations. "
                "GENERAL means ordinary questions or requests unrelated "
                "to database operations. "
                "Return only the category name."
            )
        },
        {
            "role": "user",
            "content": query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(
        teacher_model.device
    )

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()


test_queries = [
    "Show all customers from the customer table.",
    "What is the weather like today?",
    "Write an SQL query to count employees.",
    "Tell me a funny joke."
]

for query in test_queries:
    prediction = ask_teacher(query)
    print("Query:", query)
    print("Teacher prediction:", prediction)
    print("-" * 50)


Query: Show all customers from the customer table.
Teacher prediction: DATABASE
--------------------------------------------------
Query: What is the weather like today?
Teacher prediction: GENERAL
--------------------------------------------------
Query: Write an SQL query to count employees.
Teacher prediction: DATABASE
--------------------------------------------------
Query: Tell me a funny joke.
Teacher prediction: GENERAL
--------------------------------------------------


In [26]:

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

# Select 50 examples from each class
pilot_df = pd.concat([
    validation_df[validation_df["label"] == "DATABASE"].sample(
        n=50, random_state=42
    ),
    validation_df[validation_df["label"] == "GENERAL"].sample(
        n=50, random_state=42
    )
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Generate teacher predictions
predictions = []

for query in tqdm(pilot_df["text"], desc="Testing teacher"):
    prediction = ask_teacher(query).strip().upper()
    predictions.append(prediction)

pilot_df["teacher_prediction"] = predictions

# Check outputs that are not valid class labels
valid_labels = {"DATABASE", "GENERAL"}
invalid = ~pilot_df["teacher_prediction"].isin(valid_labels)

print("Total examples:", len(pilot_df))
print("Invalid outputs:", invalid.sum())

# Evaluate valid predictions
valid_df = pilot_df[~invalid]

print("\nTeacher accuracy:", round(
    accuracy_score(valid_df["label"], valid_df["teacher_prediction"]), 4
))

print("\nClassification report:")
print(classification_report(
    valid_df["label"],
    valid_df["teacher_prediction"],
    labels=["DATABASE", "GENERAL"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    valid_df["label"],
    valid_df["teacher_prediction"],
    labels=["DATABASE", "GENERAL"]
))

# Save the pilot results to Google Drive
pilot_df.to_csv(
    "/content/drive/MyDrive/Task1_Research/teacher_pilot_100.csv",
    index=False
)

print("\nPilot results saved to Google Drive.")


Testing teacher:   0%|          | 0/100 [00:00<?, ?it/s]

Total examples: 100
Invalid outputs: 0

Teacher accuracy: 0.63

Classification report:
              precision    recall  f1-score   support

    DATABASE       1.00      0.26      0.41        50
     GENERAL       0.57      1.00      0.73        50

    accuracy                           0.63       100
   macro avg       0.79      0.63      0.57       100
weighted avg       0.79      0.63      0.57       100

Confusion matrix:
[[13 37]
 [ 0 50]]

Pilot results saved to Google Drive.


In [27]:

# Show 15 DATABASE examples that the teacher misclassified
wrong_db = pilot_df[
    (pilot_df["label"] == "DATABASE") &
    (pilot_df["teacher_prediction"] != "DATABASE")
]

print("Misclassified DATABASE examples:", len(wrong_db))
print("=" * 80)

for i, (_, row) in enumerate(wrong_db.head(15).iterrows(), start=1):
    print(f"\nExample {i}")
    print("Query:", row["text"])
    print("Actual label:", row["label"])
    print("Teacher prediction:", row["teacher_prediction"])
    print("-" * 60)


Misclassified DATABASE examples: 37

Example 1
Query: How many people watched the away team of Geelong?
Actual label: DATABASE
Teacher prediction: GENERAL
------------------------------------------------------------

Example 2
Query: When the nominee(s) is john wells what is the episode for a year before 2004?
Actual label: DATABASE
Teacher prediction: GENERAL
------------------------------------------------------------

Example 3
Query: Which track has a Japanese title of メロディー?
Actual label: DATABASE
Teacher prediction: GENERAL
------------------------------------------------------------

Example 4
Query: What was the finishing time of the Stage that featured a distance of 24.00km and a start time of 21:27?
Actual label: DATABASE
Teacher prediction: GENERAL
------------------------------------------------------------

Example 5
Query: What district has Albert Rains as their representative?
Actual label: DATABASE
Teacher prediction: GENERAL
--------------------------------------------

In [28]:

def ask_teacher(query):
    messages = [
        {
            "role": "system",
            "content": """
You are a careful semantic query classifier.

Classify the user's query into exactly one label:

DATABASE:
The query is intended to retrieve, filter, compare, count,
or inspect specific records, attributes, statistics, or entities
from a structured database.
It does not need to contain SQL keywords.

GENERAL:
The query is a general conversation or request, such as greetings,
jokes, general explanations, translation, or ordinary assistance,
without a clear intent to query structured records.

Important:
- Do not classify using keywords alone.
- Consider the intended information-retrieval task.
- Do not assume every factual question is necessarily a database query.
- If the intended database-query task is unclear, use the most
  reasonable interpretation of the query itself.

Return exactly one label: DATABASE or GENERAL.
"""
        },
        {
            "role": "user",
            "content": query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip().upper()


In [29]:

new_predictions = []

for query in tqdm(pilot_df["text"], desc="Retesting teacher"):
    new_predictions.append(ask_teacher(query))

pilot_df["teacher_prediction_v2"] = new_predictions

print("Accuracy before:", round(
    accuracy_score(
        pilot_df["label"],
        pilot_df["teacher_prediction"]
    ), 4
))

print("Accuracy after:", round(
    accuracy_score(
        pilot_df["label"],
        pilot_df["teacher_prediction_v2"]
    ), 4
))

print("\nUpdated classification report:")
print(classification_report(
    pilot_df["label"],
    pilot_df["teacher_prediction_v2"],
    labels=["DATABASE", "GENERAL"],
    zero_division=0
))

print("\nPreviously incorrect DATABASE examples:")
wrong_after = pilot_df[
    (pilot_df["label"] == "DATABASE") &
    (pilot_df["teacher_prediction_v2"] != "DATABASE")
]

print("Still incorrect:", len(wrong_after), "out of 50")


Retesting teacher:   0%|          | 0/100 [00:00<?, ?it/s]

Accuracy before: 0.63
Accuracy after: 0.86

Updated classification report:
              precision    recall  f1-score   support

    DATABASE       0.97      0.74      0.84        50
     GENERAL       0.79      0.98      0.88        50

    accuracy                           0.86       100
   macro avg       0.88      0.86      0.86       100
weighted avg       0.88      0.86      0.86       100


Previously incorrect DATABASE examples:
Still incorrect: 13 out of 50


In [30]:

# Inspect DATABASE queries still misclassified by the improved prompt
wrong_db_v2 = pilot_df[
    (pilot_df["label"] == "DATABASE") &
    (pilot_df["teacher_prediction_v2"] != "DATABASE")
]

for i, (_, row) in enumerate(wrong_db_v2.iterrows(), start=1):
    print(f"\nExample {i}")
    print("Query:", row["text"])
    print("Teacher prediction:", row["teacher_prediction_v2"])
    print("-" * 60)

# Also inspect GENERAL queries incorrectly classified as DATABASE
wrong_general_v2 = pilot_df[
    (pilot_df["label"] == "GENERAL") &
    (pilot_df["teacher_prediction_v2"] != "GENERAL")
]

print("\nIncorrect GENERAL predictions:", len(wrong_general_v2))

for i, (_, row) in enumerate(wrong_general_v2.iterrows(), start=1):
    print(f"\nExample {i}")
    print("Query:", row["text"])
    print("Teacher prediction:", row["teacher_prediction_v2"])



Example 1
Query: What district has Albert Rains as their representative?
Teacher prediction: GENERAL
------------------------------------------------------------

Example 2
Query: What is the fielding team with 155 runs?
Teacher prediction: GENERAL
------------------------------------------------------------

Example 3
Query: What Per capita income has a Median family income of $50,755?
Teacher prediction: GENERAL
------------------------------------------------------------

Example 4
Query: what's the state with highest point being mount katahdin
Teacher prediction: GENERAL
------------------------------------------------------------

Example 5
Query: What was the decision on November 26?
Teacher prediction: GENERAL
------------------------------------------------------------

Example 6
Query: When 9386 is the undergraduate enrollment what is the nickname?
Teacher prediction: GENERAL
------------------------------------------------------------

Example 7
Query: Which 1991 has an A 19

In [31]:

def ask_teacher(query):
    messages = [
        {
            "role": "system",
            "content": """
You are a semantic query classifier for a database security gateway.

Classify the user's message into exactly one category.

DATABASE:
The user explicitly requests an SQL query or a database operation,
such as selecting, filtering, inserting, updating, deleting,
counting, joining, or aggregating database records.
Requests to write SQL are also DATABASE.

GENERAL:
Ordinary conversation or requests that do not explicitly ask
for SQL or a database operation, including factual questions,
greetings, jokes, explanations, translations, and general advice.

Important:
A factual question is not automatically a DATABASE query.
Do not infer a database operation just because an answer could
possibly be retrieved from a database.
Use only the intent expressed in the user's message.

Return exactly one label: DATABASE or GENERAL.
"""
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt"
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip().upper()


In [32]:

test_queries_v3 = [
    "Write an SQL query to count employees.",
    "Show all customers from the customer table.",
    "What country is Nick Price from?",
    "Tell me a funny joke.",
    "Update the salary of employee 101.",
    "What is the weather like today?",
    "Delete all records where status is inactive.",
    "Explain what machine learning means."
]

for query in test_queries_v3:
    print("Query:", query)
    print("Prediction:", ask_teacher(query))
    print("-" * 60)


Query: Write an SQL query to count employees.
Prediction: DATABASE
------------------------------------------------------------
Query: Show all customers from the customer table.
Prediction: DATABASE
------------------------------------------------------------
Query: What country is Nick Price from?
Prediction: GENERAL
------------------------------------------------------------
Query: Tell me a funny joke.
Prediction: GENERAL
------------------------------------------------------------
Query: Update the salary of employee 101.
Prediction: DATABASE
------------------------------------------------------------
Query: What is the weather like today?
Prediction: GENERAL
------------------------------------------------------------
Query: Delete all records where status is inactive.
Prediction: DATABASE
------------------------------------------------------------
Query: Explain what machine learning means.
Prediction: GENERAL
------------------------------------------------------------


In [33]:

import pandas as pd

splits = {
    "TRAIN": train_df,
    "VALIDATION": validation_df,
    "TEST": test_df
}

for split_name, df in splits.items():
    print("\n" + "=" * 65)
    print(f"{split_name} DATASET AUDIT")
    print("=" * 65)

    print("Shape:", df.shape)
    print("\nLabel counts:")
    print(df["label"].value_counts().to_string())

    print("\nMissing values:")
    print(df[["text", "label"]].isnull().sum().to_string())

    print("\nDuplicate texts:",
          df["text"].astype(str).str.strip().str.lower().duplicated().sum())

    for label in ["DATABASE", "GENERAL"]:
        subset = df[df["label"] == label]
        print(f"\n--- {label}: 5 sample queries ---")

        for query in subset["text"].head(5):
            print("-", query)



TRAIN DATASET AUDIT
Shape: (4000, 2)

Label counts:
label
DATABASE    2000
GENERAL     2000

Missing values:
text     0
label    0

Duplicate texts: 0

--- DATABASE: 5 sample queries ---
- how many coupless totalled 14.5
- Can you tell me the highest Total votes that has the % of popular vote of 0.86%, and the # of seats won larger than 0?
- What was the greatest number of Losses for the team that had 1427 Against and more than 5 Wins?
- How many picks are there for defensive back for rounds larger than 11?
- Name the average year for 4 rank

--- GENERAL: 5 sample queries ---
- what are the atmospheric conditions in sparks
- what can you tell me about miami weather
- i am needing to know the current time in the eastern timezone
- stop the current song and play the next one on this playlist
- can you translate milk into spanish for me

VALIDATION DATASET AUDIT
Shape: (799, 2)

Label counts:
label
DATABASE    400
GENERAL     399

Missing values:
text     0
label    0

Duplicate texts: 0

In [34]:

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

split_map = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

audit_results = []

for split_name, df in split_map.items():
    # Sample 20 examples from each original class
    sample_parts = []

    for label in ["DATABASE", "GENERAL"]:
        class_df = df[df["label"] == label]
        sample_parts.append(
            class_df.sample(
                n=min(20, len(class_df)),
                random_state=42
            )
        )

    sample_df = pd.concat(sample_parts).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    for _, row in tqdm(
        sample_df.iterrows(),
        total=len(sample_df),
        desc=f"Auditing {split_name}"
    ):
        prediction = ask_teacher(row["text"])

        audit_results.append({
            "split": split_name,
            "text": row["text"],
            "original_label": row["label"],
            "teacher_prediction": prediction,
            "agreement": prediction == row["label"]
        })

audit_df = pd.DataFrame(audit_results)

print("\nAgreement with original labels by split:")
print(
    audit_df.groupby("split")["agreement"]
    .agg(["sum", "count", "mean"])
    .to_string()
)

print("\nExamples where teacher and original labels disagree:")
disagreements = audit_df[
    audit_df["agreement"] == False
]

for _, row in disagreements.head(25).iterrows():
    print(f"\nSplit: {row['split']}")
    print("Query:", row["text"])
    print("Original:", row["original_label"])
    print("Teacher:", row["teacher_prediction"])
    print("-" * 50)

# Save a separate audit file; preserve the original datasets
output_path = Path(
    "/content/drive/MyDrive/Task1_Research/task1_label_audit_sample.csv"
)
audit_df.to_csv(output_path, index=False)

print("\nAudit saved to:", output_path)


Auditing train:   0%|          | 0/40 [00:00<?, ?it/s]

Auditing validation:   0%|          | 0/40 [00:00<?, ?it/s]

Auditing test:   0%|          | 0/40 [00:00<?, ?it/s]


Agreement with original labels by split:
            sum  count   mean
split                        
test         25     40  0.625
train        26     40  0.650
validation   30     40  0.750

Examples where teacher and original labels disagree:

Split: train
Query: Who was the opponent at the game at Madison Square Garden?
Original: DATABASE
Teacher: GENERAL
--------------------------------------------------

Split: train
Query: What year did University of Saskatchewan have their first season?
Original: DATABASE
Teacher: GENERAL
--------------------------------------------------

Split: train
Query: What is the position of Perpignan?
Original: DATABASE
Teacher: GENERAL
--------------------------------------------------

Split: train
Query: I want the constructor for winning driver of jim clark, pole position of graham hill and dutch grand prix
Original: DATABASE
Teacher: GENERAL
--------------------------------------------------

Split: train
Query: Where does the staff member with th

In [35]:

from pathlib import Path

rules_text = """
Task 1: Semantic Query Classification — Labeling Rules

DATABASE:
- Requests to write SQL queries.
- Requests to retrieve or filter structured database records.
- Requests to count, compare, join, or aggregate database records.
- Requests to insert, update, or delete database records.
- Natural-language questions intended to query structured records.

GENERAL:
- Greetings and ordinary conversation.
- Jokes, translations, explanations, and general advice.
- General factual questions without a clear structured-record query intent.

REVIEW:
- Ambiguous queries whose intent cannot be determined confidently.
- Malformed or incomplete queries that cannot be interpreted reliably.

Important:
- REVIEW is for manual inspection, not a final training class.
- Do not change original labels automatically based only on teacher predictions.
- Keep original datasets unchanged.
- Keep the final test labels independent from teacher-generated predictions.
"""

rules_path = Path(
    "/content/drive/MyDrive/Task1_Research/task1_labeling_rules.txt"
)

rules_path.write_text(rules_text, encoding="utf-8")

print("Labeling rules saved to:", rules_path)
print(rules_text)


Labeling rules saved to: /content/drive/MyDrive/Task1_Research/task1_labeling_rules.txt

Task 1: Semantic Query Classification — Labeling Rules

DATABASE:
- Requests to write SQL queries.
- Requests to retrieve or filter structured database records.
- Requests to count, compare, join, or aggregate database records.
- Requests to insert, update, or delete database records.
- Natural-language questions intended to query structured records.

GENERAL:
- Greetings and ordinary conversation.
- Jokes, translations, explanations, and general advice.
- General factual questions without a clear structured-record query intent.

REVIEW:
- Ambiguous queries whose intent cannot be determined confidently.
- Malformed or incomplete queries that cannot be interpreted reliably.

Important:
- REVIEW is for manual inspection, not a final training class.
- Do not change original labels automatically based only on teacher predictions.
- Keep original datasets unchanged.
- Keep the final test labels independ

In [36]:

# Keep only cases where teacher and original labels disagree
review_candidates = audit_df[
    audit_df["agreement"] == False
].copy()

# Add blank columns for independent manual review
review_candidates["reviewed_label"] = ""
review_candidates["review_notes"] = ""

# Save separately; do not overwrite original datasets
review_path = (
    "/content/drive/MyDrive/Task1_Research/"
    "task1_manual_review_candidates.csv"
)

review_candidates.to_csv(review_path, index=False)

print("Candidates for manual review:", len(review_candidates))
print("Saved to:", review_path)

print("\nCandidates by split:")
print(review_candidates["split"].value_counts().to_string())

print("\nFirst 10 candidates:")
display(
    review_candidates[
        ["split", "text", "original_label",
         "teacher_prediction", "reviewed_label"]
    ].head(10)
)


Candidates for manual review: 39
Saved to: /content/drive/MyDrive/Task1_Research/task1_manual_review_candidates.csv

Candidates by split:
split
test          15
train         14
validation    10

First 10 candidates:


,split,text,original_label,teacher_prediction,reviewed_label
0,train,Who was the opponent at the game at Madison Sq...,DATABASE,GENERAL,
1,train,What year did University of Saskatchewan have ...,DATABASE,GENERAL,
4,train,What is the position of Perpignan?,DATABASE,GENERAL,
5,train,I want the constructor for winning driver of j...,DATABASE,GENERAL,
9,train,Where does the staff member with the first nam...,DATABASE,GENERAL,
12,train,Which region is Coach Trevor Gleeson in?,DATABASE,GENERAL,
16,train,"Which catalog was published on December 19, 2001?",DATABASE,GENERAL,
18,train,Name the loss when the record is 76-81,DATABASE,GENERAL,
20,train,What is the language of the film originally ti...,DATABASE,GENERAL,
22,train,What position does the player from arkansas play?,DATABASE,GENERAL,


In [37]:

# Use the audit results already created
train_val_review = review_candidates[
    review_candidates["split"].isin(["train", "validation"])
].copy()

review_path = (
    "/content/drive/MyDrive/Task1_Research/"
    "task1_train_validation_review.csv"
)

train_val_review.to_csv(review_path, index=False)

print("Training candidates:",
      (train_val_review["split"] == "train").sum())

print("Validation candidates:",
      (train_val_review["split"] == "validation").sum())

print("Total candidates for review:", len(train_val_review))
print("Saved to:", review_path)


Training candidates: 14
Validation candidates: 10
Total candidates for review: 24
Saved to: /content/drive/MyDrive/Task1_Research/task1_train_validation_review.csv


In [38]:

teacher_prompt_tests = [
    ("What country is Nick Price from?", "DATABASE"),
    ("What was the score when the home team was Shrewsbury Town?", "DATABASE"),
    ("Write an SQL query to count employees.", "DATABASE"),
    ("Update the salary of employee 101.", "DATABASE"),
    ("Tell me a funny joke.", "GENERAL"),
    ("Translate milk into Spanish.", "GENERAL"),
    ("What is the weather like today?", "GENERAL"),
    ("Explain what machine learning means.", "GENERAL"),
]

for query, expected in teacher_prompt_tests:
    prediction = ask_teacher(query)

    print("Query:", query)
    print("Expected:", expected)
    print("Predicted:", prediction)
    print("Match:", prediction == expected)
    print("-" * 60)


Query: What country is Nick Price from?
Expected: DATABASE
Predicted: GENERAL
Match: False
------------------------------------------------------------
Query: What was the score when the home team was Shrewsbury Town?
Expected: DATABASE
Predicted: GENERAL
Match: False
------------------------------------------------------------
Query: Write an SQL query to count employees.
Expected: DATABASE
Predicted: DATABASE
Match: True
------------------------------------------------------------
Query: Update the salary of employee 101.
Expected: DATABASE
Predicted: DATABASE
Match: True
------------------------------------------------------------
Query: Tell me a funny joke.
Expected: GENERAL
Predicted: GENERAL
Match: True
------------------------------------------------------------
Query: Translate milk into Spanish.
Expected: GENERAL
Predicted: GENERAL
Match: True
------------------------------------------------------------
Query: What is the weather like today?
Expected: GENERAL
Predicted: GENER

In [39]:

def ask_teacher(query):
    messages = [
        {
            "role": "system",
            "content": """
You classify queries for a database security gateway.

Return exactly one label: DATABASE or GENERAL.

DATABASE:
- Explicit SQL requests or database operations.
- Requests to retrieve specific records, entities, attributes,
  statistics, scores, dates, positions, or other fields from
  structured data.
- Natural-language questions that appear to query structured
  records, including questions with filtering conditions,
  comparisons, counts, rankings, or specific record attributes.

GENERAL:
- Greetings, jokes, translations, ordinary conversation,
  explanations, and general advice.
- Broad conversational questions that do not appear to request
  information from specific structured records.

Important:
- A DATABASE query does not need SQL keywords.
- Do not automatically classify every factual question as GENERAL.
- Consider whether the query seeks a specific attribute or result
  that could naturally be retrieved from structured records.
- Classify according to the query's expressed intent, not its
  wording style alone.

Return only DATABASE or GENERAL.
"""
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt"
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip().upper()


In [40]:

teacher_prompt_tests = [
    ("What country is Nick Price from?", "DATABASE"),
    ("What was the score when the home team was Shrewsbury Town?", "DATABASE"),
    ("Write an SQL query to count employees.", "DATABASE"),
    ("Update the salary of employee 101.", "DATABASE"),
    ("Tell me a funny joke.", "GENERAL"),
    ("Translate milk into Spanish.", "GENERAL"),
    ("What is the weather like today?", "GENERAL"),
    ("Explain what machine learning means.", "GENERAL"),
]

for query, expected in teacher_prompt_tests:
    prediction = ask_teacher(query)

    print("Query:", query)
    print("Expected:", expected)
    print("Predicted:", prediction)
    print("Match:", prediction == expected)
    print("-" * 60)


Query: What country is Nick Price from?
Expected: DATABASE
Predicted: GENERAL
Match: False
------------------------------------------------------------
Query: What was the score when the home team was Shrewsbury Town?
Expected: DATABASE
Predicted: DATABASE
Match: True
------------------------------------------------------------
Query: Write an SQL query to count employees.
Expected: DATABASE
Predicted: DATABASE
Match: True
------------------------------------------------------------
Query: Update the salary of employee 101.
Expected: DATABASE
Predicted: DATABASE
Match: True
------------------------------------------------------------
Query: Tell me a funny joke.
Expected: GENERAL
Predicted: GENERAL
Match: True
------------------------------------------------------------
Query: Translate milk into Spanish.
Expected: GENERAL
Predicted: GENERAL
Match: True
------------------------------------------------------------
Query: What is the weather like today?
Expected: GENERAL
Predicted: GENER

In [1]:
from transformers import AutoTokenizer

teacher_model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    teacher_model_name
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [2]:
for label in ["DATABASE", "GENERAL"]:
    token_ids = tokenizer.encode(
        label,
        add_special_tokens=False
    )

    print("Label:", label)
    print("Token IDs:", token_ids)
    print("Number of tokens:", len(token_ids))
    print("-" * 40)

Label: DATABASE
Token IDs: [46499]
Number of tokens: 1
----------------------------------------
Label: GENERAL
Token IDs: [94736]
Number of tokens: 1
----------------------------------------


In [3]:
%pip install -U "bitsandbytes>=0.46.1"

In [4]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

teacher_model_name = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

teacher_model.eval()

print("Teacher model loaded successfully!")

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher model loaded successfully!


In [5]:
import torch
import torch.nn.functional as F

def get_teacher_soft_targets(query, temperature=2.0):

    prompt = f"""You classify queries for a database security gateway.

Return exactly one label: DATABASE or GENERAL.

DATABASE: A query requesting SQL operations or information from structured database records.
GENERAL: Ordinary conversation, jokes, translations, explanations, or general advice.

Query: {query}
Label:"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model(
            input_ids=inputs,
            use_cache=False
        )

    # Get logits for the next token
    next_token_logits = outputs.logits[0, -1, :].float()

    # Get the token IDs for the two class labels
    database_id = tokenizer.encode(
        "DATABASE", add_special_tokens=False
    )[0]

    general_id = tokenizer.encode(
        "GENERAL", add_special_tokens=False
    )[0]

    # Extract the two class logits
    class_logits = torch.stack([
        next_token_logits[database_id],
        next_token_logits[general_id]
    ])

    # Convert logits into soft-target probabilities
    probabilities = F.softmax(
        class_logits / temperature,
        dim=0
    )

    return {
        "DATABASE": probabilities[0].item(),
        "GENERAL": probabilities[1].item()
    }

In [7]:
import torch
import torch.nn.functional as F

def get_teacher_soft_targets(query, temperature=2.0):

    prompt = f"""You classify queries for a database security gateway.

Return exactly one label: DATABASE or GENERAL.

DATABASE: A query requesting SQL operations or information from structured database records.
GENERAL: Ordinary conversation, jokes, translations, explanations, or general advice.

Query: {query}
Label:"""

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move input tensors to the teacher model's device
    inputs = inputs.to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model(
            **inputs,
            use_cache=False
        )

    # Logits for the next generated token
    next_token_logits = outputs.logits[0, -1, :].float()

    database_id = tokenizer.encode(
        "DATABASE", add_special_tokens=False
    )[0]

    general_id = tokenizer.encode(
        "GENERAL", add_special_tokens=False
    )[0]

    class_logits = torch.stack([
        next_token_logits[database_id],
        next_token_logits[general_id]
    ])

    probabilities = F.softmax(
        class_logits / temperature,
        dim=0
    )

    return {
        "DATABASE": probabilities[0].item(),
        "GENERAL": probabilities[1].item()
    }

In [9]:
result = get_teacher_soft_targets(
    "Write an SQL query to count employees."
)

print("Soft targets:", result)
print("Probability sum:", sum(result.values()))

Soft targets: {'DATABASE': 0.999976396560669, 'GENERAL': 2.3552616767119616e-05}
Probability sum: 0.9999999491774361
